# 제주 특산물 가격 예측 - LSTM 시계열 모델

| 항목 | 내용 |
|------|------|
| **버전** | v2.0.0 |
| **날짜** | 2026-03-12 |
| **모델** | Keras LSTM (Bidirectional) |
| **전처리** | v1.0.1 동일 (DACON 1위 전처리) |
| **핵심** | 그룹별 슬라이딩 윈도우 시퀀스 + 오토레그레시브 테스트 예측 |
| **출력** | results/submission_v2.0.0.csv |

## v2.0.0 특징
| 항목 | 내용 |
|------|------|
| 시퀀스 길이 | LOOK_BACK = 14일 |
| 입력 피처 | 시간 피처 + 그룹 one-hot + price_transformed (오토레그레시브) |
| 모델 구조 | LSTM(128, return_seq) → Dropout → LSTM(64) → Dropout → Dense(32) → Dense(1) |
| 타겟 스케일 | feature_scaler + target_scaler (별도 StandardScaler) |
| 테스트 예측 | 그룹별 마지막 14일 seed → 오토레그레시브 순차 예측 |

---
## 1. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, datetime, random, os, platform
warnings.filterwarnings('ignore')

import holidays

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

SEED = 2024
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU 사용: {bool(tf.config.list_physical_devices("GPU"))}')

---
## 2. 데이터 로드

In [ ]:
DATA_PATH = './data/'

train = pd.read_csv(DATA_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATA_PATH + 'test.csv',  encoding='utf-8-sig')
sub   = pd.read_csv(DATA_PATH + 'sample_submission.csv', encoding='utf-8-sig')

print(f'train: {train.shape}, test: {test.shape}')
train.head(3)

---
## 3. 전처리 (v1.0.1 동일)

In [ ]:
def pre_all(train, test):
    train['timestamp'] = pd.to_datetime(train['timestamp'])
    test['timestamp']  = pd.to_datetime(test['timestamp'])
    df = pd.concat([train, test]).reset_index(drop=True)
    df.rename(columns={'supply(kg)': 'supply', 'price(원/kg)': 'price'}, inplace=True)

    df['year']     = df['timestamp'].dt.year
    df['month']    = df['timestamp'].dt.month
    df['day']      = df['timestamp'].dt.day
    df['week_day'] = df['timestamp'].dt.weekday

    le = LabelEncoder()
    df['year_month'] = df['timestamp'].map(lambda x: f'{x.year}-{x.month}')
    df['year_month'] = le.fit_transform(df['year_month'])

    df['week'] = df['timestamp'].map(
        lambda x: datetime.datetime(x.year, x.month, x.day).isocalendar()[1]
    )
    week_offsets = {2019: 0, 2020: 52, 2021: 52+53, 2022: 52+53+53, 2023: 52+53+53+52}
    df['week_num'] = df.apply(lambda r: int(r['week']) + week_offsets.get(r['year'], 0), axis=1)
    df.loc[df['timestamp'] == '2019-12-30', 'week_num'] = 52
    df.loc[df['timestamp'] == '2019-12-31', 'week_num'] = 52

    kr_holi = holidays.KR()
    df['holiday'] = df['timestamp'].map(lambda x: 1 if x in kr_holi else 0)

    train_out = df[~df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    test_out  = df[ df['price'].isnull()].sort_values('timestamp').reset_index(drop=True)
    print(f'전처리 완료 — train: {train_out.shape}, test: {test_out.shape}')
    return train_out, test_out

train_pre, test_pre = pre_all(train, test)

---
## 4. 이상치 처리 + 타겟 변환

In [ ]:
# 이상치: 품목별 임계값 초과 → 평균 대체
outlier_thresholds = {'TG': 20000, 'RD': 5000, 'BC': 8000, 'CB': 2300}
for item, thr in outlier_thresholds.items():
    idx = train_pre[(train_pre['item'] == item) & (train_pre['price'] > thr)].index
    if len(idx):
        mean_p = train_pre[(train_pre['item'] == item) & (train_pre['price'] != 0)]['price'].mean()
        train_pre.loc[idx, 'price'] = mean_p
        print(f'{item}: {len(idx)}개 이상치 → 평균({mean_p:.0f})')

# 타겟 변환 (TG: sqrt, 비TG: log1p)
train_pre['is_tg'] = (train_pre['item'] == 'TG').astype(int)
test_pre['is_tg']  = (test_pre['item']  == 'TG').astype(int)
train_pre['price_transformed'] = np.where(
    train_pre['is_tg'] == 1,
    np.sqrt(train_pre['price']),
    np.log1p(train_pre['price'])
)

# TG 공휴일 보정
tg_mask = (train_pre['item'] == 'TG') & (train_pre['holiday'] == 1) & (train_pre['price'] != 0)
active_holi = list(train_pre[tg_mask].groupby('timestamp').count().reset_index()['timestamp'])
fix_idx = train_pre[train_pre['timestamp'].isin(active_holi)].index
train_pre.loc[fix_idx, 'holiday'] = 0
print(f'TG 공휴일 보정: {len(fix_idx)}개')
print(f'price_transformed 범위: [{train_pre["price_transformed"].min():.2f}, {train_pre["price_transformed"].max():.2f}]')

---
## 5. 피처 인코딩 (시퀀스용 DataFrame 구성)

In [ ]:
GROUP_COLS       = ['item', 'corporation', 'location']
TIME_FEAT_COLS   = ['year_month', 'week_num', 'week_day', 'month', 'day', 'holiday']
PRICE_COL        = 'price_transformed'
DROP_COLS_COMMON = ['supply', 'week', 'is_tg', 'price']

def build_encoded(df, group_cols, drop_cols):
    """one-hot 인코딩 후 원본 group_cols 문자열 복원"""
    enc = pd.get_dummies(df.drop(columns=[c for c in drop_cols if c in df.columns]),
                         columns=group_cols, dtype=float)
    for col in group_cols:
        enc[col] = df[col].values
    return enc

train_enc = build_encoded(train_pre, GROUP_COLS, DROP_COLS_COMMON)
test_enc  = build_encoded(test_pre,  GROUP_COLS, DROP_COLS_COMMON)

# 컬럼 정렬 (train 기준)
CAT_FEAT_COLS = sorted([c for c in train_enc.columns
                         if any(c.startswith(p + '_') for p in GROUP_COLS)])
for col in CAT_FEAT_COLS:
    if col not in test_enc.columns:
        test_enc[col] = 0.0

NON_PRICE_FEAT_COLS = TIME_FEAT_COLS + CAT_FEAT_COLS
SEQ_FEAT_COLS       = NON_PRICE_FEAT_COLS + [PRICE_COL]

print(f'SEQ_FEAT_COLS: {len(SEQ_FEAT_COLS)}개')
print(f'TIME: {len(TIME_FEAT_COLS)}, CAT: {len(CAT_FEAT_COLS)}, PRICE: 1')

---
## 6. 시퀀스 생성 (슬라이딩 윈도우, 그룹별)

In [ ]:
LOOK_BACK = 14  # 2주 히스토리

def make_sequences(df, group_cols, feat_cols, price_col, look_back):
    """
    그룹별 슬라이딩 윈도우 시퀀스 생성
    X[i] = feat_cols[i-look_back : i]  → (look_back, n_feat)
    y[i] = price_col[i]                → scalar
    """
    all_X, all_y, all_is_tg = [], [], []

    for name, grp in df.groupby(group_cols):
        grp = grp.sort_values('timestamp').reset_index(drop=True)
        if len(grp) <= look_back:
            continue

        X_grp = grp[feat_cols].values.astype(np.float32)
        y_grp = grp[price_col].values.astype(np.float32)
        is_tg = int(name[0] == 'TG')

        for i in range(look_back, len(grp)):
            all_X.append(X_grp[i - look_back : i])  # (look_back, n_feat)
            all_y.append(y_grp[i])
            all_is_tg.append(is_tg)

    return (np.array(all_X,     dtype=np.float32),
            np.array(all_y,     dtype=np.float32),
            np.array(all_is_tg, dtype=np.int32))


X_seq, y_seq, is_tg_seq = make_sequences(
    train_enc, GROUP_COLS, SEQ_FEAT_COLS, PRICE_COL, LOOK_BACK
)
print(f'X_seq: {X_seq.shape}  (샘플, 시퀀스길이, 피처수)')
print(f'y_seq: {y_seq.shape}')

---
## 7. 학습/검증 분리 + 스케일링

In [ ]:
# 시간순 8:2 분리
split = int(len(X_seq) * 0.8)
X_tr, X_vl   = X_seq[:split], X_seq[split:]
y_tr, y_vl   = y_seq[:split], y_seq[split:]
is_tg_vl     = is_tg_seq[split:]

# Feature scaler: fit on train (flatten to 2D)
n_tr, lbk, nf = X_tr.shape
feat_scaler = StandardScaler()
feat_scaler.fit(X_tr.reshape(-1, nf))
X_tr_scaled = feat_scaler.transform(X_tr.reshape(-1, nf)).reshape(n_tr, lbk, nf)
X_vl_scaled = feat_scaler.transform(X_vl.reshape(-1, nf)).reshape(X_vl.shape)

# Target scaler: 별도 StandardScaler
target_scaler = StandardScaler()
y_tr_scaled = target_scaler.fit_transform(y_tr.reshape(-1, 1)).flatten()
y_vl_scaled = target_scaler.transform(y_vl.reshape(-1, 1)).flatten()

print(f'학습: {X_tr_scaled.shape}, 검증: {X_vl_scaled.shape}')
print(f'y_tr_scaled: mean={y_tr_scaled.mean():.3f}, std={y_tr_scaled.std():.3f}')

---
## 8. LSTM 모델 설계

```
Input(14, N피처)
  → LSTM(128, return_sequences=True) → Dropout(0.2)
  → LSTM(64)                         → Dropout(0.2)
  → Dense(32, relu)
  → Dense(1)
```

In [ ]:
def build_lstm(input_shape, lr=1e-3):
    inp = keras.Input(shape=input_shape)
    x = layers.LSTM(128, return_sequences=True)(inp)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(64)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1)(x)

    model = keras.Model(inp, out, name='jeju_lstm_v2')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mae',
        metrics=['mse']
    )
    return model

model = build_lstm((LOOK_BACK, len(SEQ_FEAT_COLS)))
model.summary()

---
## 9. 학습

In [ ]:
os.makedirs('./models',  exist_ok=True)
os.makedirs('./results', exist_ok=True)

MODEL_PATH = './models/jeju_lstm_v2.0.0.keras'

cb_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=20,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                patience=8, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_loss',
                              save_best_only=True, verbose=0)
]

history = model.fit(
    X_tr_scaled, y_tr_scaled,
    validation_data=(X_vl_scaled, y_vl_scaled),
    epochs=200,
    batch_size=256,
    callbacks=cb_list,
    verbose=1
)

best_epoch = np.argmin(history.history['val_loss'])
print(f'\nBest Epoch: {best_epoch}, Best Val MAE: {min(history.history["val_loss"]):.4f}')

---
## 10. 학습 곡선

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train MAE', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val MAE',   color='darkorange', ls='--')
axes[0].axvline(best_epoch, color='red', ls=':', alpha=0.7, label=f'Best({best_epoch})')
axes[0].set_title('MAE 학습 곡선', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history.history['mse'],     label='Train MSE', color='steelblue')
axes[1].plot(history.history['val_mse'], label='Val MSE',   color='darkorange', ls='--')
axes[1].set_title('MSE 학습 곡선', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('LSTM v2.0.0 학습 곡선', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/training_curve_v2.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. 모델 평가

In [ ]:
y_pred_scaled = model.predict(X_vl_scaled, verbose=0).flatten()
y_pred_transformed = target_scaler.inverse_transform(
    y_pred_scaled.reshape(-1, 1)
).flatten()

# 역변환: TG → 제곱, 비TG → expm1
y_pred_orig = np.where(
    is_tg_vl.astype(bool),
    np.power(np.clip(y_pred_transformed, 0, None), 2),
    np.expm1(y_pred_transformed)
)
y_pred_orig = np.clip(y_pred_orig, 0, None)

# 검증 원본 가격 복원
y_vl_orig = np.where(
    is_tg_vl.astype(bool),
    np.power(np.clip(y_vl, 0, None), 2),
    np.expm1(y_vl)
)
y_vl_orig = np.clip(y_vl_orig, 0, None)

mae  = mean_absolute_error(y_vl_orig, y_pred_orig)
rmse = np.sqrt(mean_squared_error(y_vl_orig, y_pred_orig))
r2   = r2_score(y_vl_orig, y_pred_orig)
mape = np.mean(np.abs((y_vl_orig - y_pred_orig) / (y_vl_orig + 1e-8))) * 100

print('=' * 50)
print('  LSTM v2.0.0 검증 성능')
print('=' * 50)
print(f'  MAE  : {mae:>10,.2f} 원/kg')
print(f'  RMSE : {rmse:>10,.2f} 원/kg')
print(f'  R²   : {r2:>10.4f}')
print(f'  MAPE : {mape:>10.2f} %')
print('=' * 50)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

max_v = max(y_vl_orig.max(), y_pred_orig.max())
axes[0].scatter(y_vl_orig, y_pred_orig, alpha=0.3, s=8, color='steelblue')
axes[0].plot([0, max_v], [0, max_v], 'r--', lw=2)
axes[0].set_title(f'실제 vs 예측  R²={r2:.4f}', fontsize=12)
axes[0].set_xlabel('실제'); axes[0].set_ylabel('예측'); axes[0].grid(alpha=0.3)

res = y_vl_orig - y_pred_orig
axes[1].hist(res, bins=50, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=2, ls='--')
axes[1].set_title(f'잔차 분포  mean={res.mean():.1f}', fontsize=12)
axes[1].set_xlabel('잔차'); axes[1].grid(alpha=0.3)

n = min(300, len(y_vl_orig))
axes[2].plot(y_vl_orig[:n],   label='실제', lw=1.2)
axes[2].plot(y_pred_orig[:n], label='예측', lw=1.2, ls='--')
axes[2].set_title(f'실제 vs 예측 (처음 {n}개)', fontsize=12)
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('LSTM v2.0.0 검증 결과', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('./results/eval_plots_v2.0.0.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 12. 테스트 예측 (오토레그레시브)

> 각 그룹(item × corp × loc)별로:
> 1. 학습 데이터 마지막 14일을 seed로 사용
> 2. 테스트 기간을 순차적으로 예측 (이전 예측값을 다음 입력의 price_transformed로 활용)

In [ ]:
def predict_autoregressive(model, train_enc, test_enc, group_cols,
                            non_price_feat_cols, price_col,
                            feat_scaler, target_scaler, look_back):
    seq_feat_cols = non_price_feat_cols + [price_col]
    nf = len(seq_feat_cols)
    results = []

    for name, test_grp in test_enc.groupby(group_cols):
        test_grp = test_grp.sort_values('timestamp').reset_index(drop=True)
        is_tg = (name[0] == 'TG')

        # 해당 그룹의 학습 seed
        mask = pd.Series(True, index=train_enc.index)
        for col, val in zip(group_cols, name):
            mask = mask & (train_enc[col] == val)
        train_grp = train_enc[mask].sort_values('timestamp').tail(look_back)

        if len(train_grp) == 0:
            for _, row in test_grp.iterrows():
                results.append({'ID': row['ID'], 'answer': 0.0})
            continue

        # seed 패딩 (학습 데이터가 look_back보다 짧은 경우)
        seed_vals = train_grp[seq_feat_cols].values.astype(np.float32)
        if len(seed_vals) < look_back:
            n_pad = look_back - len(seed_vals)
            pad = np.tile(seed_vals[0], (n_pad, 1))
            seed_vals = np.vstack([pad, seed_vals])

        current_seq = feat_scaler.transform(seed_vals)  # (look_back, nf)

        for _, row in test_grp.iterrows():
            # 예측
            X_in = current_seq[np.newaxis].astype(np.float32)  # (1, look_back, nf)
            pred_scaled = model.predict(X_in, verbose=0)[0, 0]
            pred_transformed = float(
                target_scaler.inverse_transform([[pred_scaled]])[0, 0]
            )

            # 원본 가격 역변환
            if is_tg:
                pred_price = max(0.0, pred_transformed) ** 2
            else:
                pred_price = float(np.expm1(pred_transformed))
            pred_price = max(0.0, pred_price)
            results.append({'ID': row['ID'], 'answer': pred_price})

            # 다음 스텝 입력 구성
            new_row = np.zeros(nf, dtype=np.float32)
            for j, col in enumerate(seq_feat_cols):
                if col == price_col:
                    new_row[j] = pred_transformed
                elif col in row.index and not pd.isna(row[col]):
                    new_row[j] = float(row[col])
            new_row_scaled = feat_scaler.transform(new_row.reshape(1, -1))
            current_seq = np.vstack([current_seq[1:], new_row_scaled])

    return pd.DataFrame(results)


print('오토레그레시브 예측 시작...')
test_preds = predict_autoregressive(
    model, train_enc, test_enc,
    GROUP_COLS, NON_PRICE_FEAT_COLS, PRICE_COL,
    feat_scaler, target_scaler, LOOK_BACK
)
print(f'예측 완료: {len(test_preds)}행')
test_preds.head()

---
## 13. 후처리 + 제출 파일 생성

In [ ]:
# 품목 정보 복원
test_preds = test_preds.merge(
    test_pre[['ID', 'item']].drop_duplicates(), on='ID', how='left'
)

# 후처리: 품목별 최솟값 미만 → 0
min_thresholds = {'TG': 400, 'CB': 50, 'RD': 10, 'CR': 150, 'BC': 100}
for item, thr in min_thresholds.items():
    mask = (test_preds['item'] == item) & (test_preds['answer'] < thr)
    test_preds.loc[mask, 'answer'] = 0.0
    n = mask.sum()
    if n > 0:
        print(f'{item}: {n}개 → 0 처리 (임계값: {thr})')

print('\n예측 통계:')
print(test_preds.groupby('item')['answer'].agg(['mean', 'min', 'max']).round(1))

# 제출 파일
result = sub[['ID']].merge(test_preds[['ID', 'answer']], on='ID', how='left')
result['answer'] = result['answer'].fillna(0.0)

SUBMISSION_PATH = './results/submission_v2.0.0.csv'
result.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')
print(f'\n저장: {SUBMISSION_PATH}, 행: {len(result)}, 결측: {result["answer"].isnull().sum()}')
result.head(10)

---
## 14. 결과 요약

In [ ]:
print('=' * 60)
print('  LSTM v2.0.0 최종 결과')
print('=' * 60)
print(f'  시퀀스 길이 (LOOK_BACK) : {LOOK_BACK}')
print(f'  입력 피처 수            : {len(SEQ_FEAT_COLS)}개')
print(f'  학습 시퀀스             : {X_tr.shape[0]:,}개')
print(f'  검증 시퀀스             : {X_vl.shape[0]:,}개')
print(f'  Best Epoch              : {best_epoch}')
print()
print('  [검증 성능]')
print(f'  MAE  = {mae:>10,.2f} 원/kg')
print(f'  RMSE = {rmse:>10,.2f} 원/kg')
print(f'  R²   = {r2:>10.4f}')
print(f'  MAPE = {mape:>10.2f} %')
print()
print(f'  제출 파일: {SUBMISSION_PATH}')
print(f'  모델 파일: {MODEL_PATH}')
print('=' * 60)

### 다음 버전

| 버전 | 개선 내용 |
|------|----------|
| **v2.1.0** | GRU 모델 비교 |
| **v2.2.0** | Attention 메커니즘 추가 |
| **v3.0.0** | DNN + CatBoost + XGBoost 앙상블 |